# Cruces entre datasets (caso 1.3)

El caso 1.3 pide **agrupar los distritos de Madrid según su perfil de riesgo y su contexto social**, cruzando el socioeconómico con las activaciones (por ejemplo, ver si la renta media se relaciona con ciertos tipos de emergencia, o si la densidad influye en los tiempos de llegada). Bluetab nos pidió además mirar qué otros datasets se pueden cruzar para sacar más información.

Aquí revisamos dataset a dataset qué cruces son posibles, cuáles usamos y por qué, y montamos la tabla final: **una fila por distrito**, que es la que entra en el clustering.

**Orden:** se ejecuta después de `limpieza_socioeconomico.ipynb`, `limpieza_distritos.ipynb` y `limpieza_activaciones.ipynb`, porque usa los CSV limpios que generan.

### Trazabilidad de los correos citados y alcance real

Se releyeron las fuentes primarias de la cuenta UC3M el 17/11/2026. El mensaje recibido de `cruz.mateo@bluetab.net`, del 14/10/2026 a las 12:07 CEST, asunto de seguimiento del proyecto, ID `1a139e1c233cd6d2`, respalda la ausencia de día, las equivalencias de prioridad, el rango 0-115 y la distinción de variantes explícitas de Vallecas frente a los 39 registros ambiguos de origen. Describe mismo día salvo cruce de medianoche, pero NO fija nuestras ventanas 20:00-23:59 y 00:00-06:59: esas ventanas y su cuantificación son decisión propia basada en la distribución observada. El mensaje permite más de una solución para Vallecas; elegimos no imputar, con el motivo explicado aquí. No convierte la cifra de origen en un recuento final.

El mismo correo exige un criterio consistente para corregir el catálogo, pero NO aprueba cada equivalencia clínica, categoría, prioridad ni tiempo elegido después. Esas decisiones son nuestras y se justifican en la limpieza. El mensaje de `david.millan@bluetab.net`, 14/10/2026 a las 12:21, ID `1a139eeab43196b9`, solicita propuestas y justificación, no es aprobación de una solución concreta.

El mensaje recibido de `maria.angeles.quesada@bluetab.net`, 15/09/2026 a las 17:55 CEST, ID `1a0a5c9f014196b8`, comunica el envío del dataset y planteamiento del proyecto. Esto respalda la fecha de entrega, NO el año de observación de cada catálogo ni una supuesta instantánea homogénea de 2024.

Las anotaciones históricas que atribuyen al cliente la prohibición de diccionarios cerrados no tienen aquí una fuente primaria concreta recuperada. Se conservan como anotación histórica no verificada; no se usan como autoridad metodológica. La regla de normalización se defiende por ser explícita, reproducible y no forzar equivalencias ambiguas.


El alcance exacto de las fuentes primarias está identificado al inicio. Las ventanas concretas de medianoche y la decisión de no imputar son propias; no se atribuye al correo la aprobación de cada valor del catálogo.

In [1]:
import pandas as pd
import numpy as np
import unicodedata
from pathlib import Path

carpeta = Path('.') if Path('tabla_distritos.csv').is_file() else Path('1_3_clustering')
raiz = carpeta.resolve().parent

def cargar(nombre):
    candidatas = [carpeta / nombre, raiz / 'data_clean' / nombre, raiz / 'data' / nombre]
    ruta = next((p for p in candidatas if p.is_file()), None)
    if ruta is None:
        raise FileNotFoundError(nombre)
    return pd.read_csv(ruta)

def normalizar_texto(valor):
    if pd.isna(valor):
        return None
    s = unicodedata.normalize('NFD', str(valor).strip().upper())
    return ''.join(c for c in s if unicodedata.category(c) != 'Mn')

soc = cargar('socioeconomico_limpio.csv')
dis = cargar('distritos_limpio.csv')
act = cargar('activaciones_limpio.csv')

print(f"socioeconomico: {soc.shape}, distritos: {dis.shape}, activaciones: {act.shape}")

socioeconomico: (21, 15), distritos: (21, 7), activaciones: (146567, 13)


## 1. Socioeconómico + distritos: la base

Las dos tablas tienen una fila por distrito y el mismo `distrito_id`, así que se unen directo y encajan las 21 de 21. Es la tabla maestra: de aquí salen población, superficie y densidad, coherentes con los indicadores sociales.

In [2]:
# Quitamos del socioeconomico las columnas que ya estan en distritos (poblacion y superficie),
# porque son las mismas (se comprobo en la limpieza) y si no saldrian duplicadas.
base = dis.merge(soc.drop(columns=['poblacion_2024', 'superficie_km2']), on='distrito_id')
print(f"Filas tras unir: {len(base)} (de 21)")
print(f"Distritos sin pareja: {base['distrito'].isna().sum()}")
base[['distrito_id', 'nombre', 'poblacion', 'superficie_km2', 'renta_media_hogar',
      'indice_vulnerabilidad', 'densidad_hab_km2']].head()

Filas tras unir: 21 (de 21)
Distritos sin pareja: 0


,distrito_id,nombre,poblacion,superficie_km2,renta_media_hogar,indice_vulnerabilidad,densidad_hab_km2
0,1,CENTRO,149718,5.23,42831.0,0.35,28626.8
1,2,ARGANZUELA,155115,6.47,39876.0,0.42,23974.5
2,3,RETIRO,119915,5.46,61604.0,0.22,21962.5
3,4,SALAMANCA,147707,5.41,71432.0,0.15,27302.6
4,5,CHAMARTIN,146512,9.19,62145.0,0.18,15942.5


## 2. Activaciones por distrito: la demanda

Las activaciones ya traen el distrito normalizado de la limpieza. Quedan fuera del análisis por distrito las 567 filas con distrito `NO CONSTA` (0,4%) y las 34 de fuera del término municipal (33 `FUERA DE TERMINO` y 1 `CARRETERAS Y CIRCUNVALACIONES`). Siguen en el CSV limpio y cuentan en la demanda total de Madrid, pero no pertenecen a ningún distrito de la ciudad.

Dos cosas han cambiado con la última versión de la limpieza (`limpieza_activaciones.ipynb`):

- `tipo_emergencia` y `prioridad` ya vienen del **catálogo de tipos** (39 códigos): la prioridad es la que el catálogo asigna a cada tipo de emergencia (lo grave que suele ser ese tipo), **no** la que se registró en cada evento. Las dos solo coincidían en el 37,7% de las filas, así que `pct_prioridad_alta` mide ahora el mix de gravedad típica de lo que se atiende en cada distrito, no un registro caso a caso.
- El tiempo de respuesta se llama `tiempo_de_respuesta` (antes `tiempo_respuesta_min`), para no confundirlo con `tiempo_respuesta_objetivo_min` del catálogo. Se calculó en la limpieza con la regla de medianoche que pidió Bluetab (mismo día salvo cruce de 20:00-23:59 a 00:00-06:59, que suma 24 h; las incoherencias quedan vacías sin borrar la activación).

Agrupamos por distrito y calculamos:

- **Nº de activaciones** y **activaciones por 1.000 habitantes**. Dividimos por población porque si no, los distritos grandes siempre parecen peores solo por tener más gente.
- **% de prioridad alta del catálogo**, como indicador de la gravedad típica de la demanda.
- **Edad media de los pacientes**, que se puede comparar con la edad media del distrito.
- **Tiempo medio de respuesta**.

In [3]:
# Solo los 21 distritos de Madrid ciudad: fuera NO CONSTA y las activaciones de fuera del termino
act_distrito = act[act['distrito'].isin(set(dis['nombre']))].copy()
print(f"Activaciones con distrito asignado: {len(act_distrito)} de {len(act)} "
      f"(quedan fuera {len(act) - len(act_distrito)}, el "
      f"{(len(act) - len(act_distrito)) / len(act) * 100:.2f}%)")

# La prioridad es la del catalogo (gravedad tipica del tipo), en minusculas: alta/media/baja
demanda = act_distrito.groupby('distrito').agg(
    n_activaciones=('activacion_id', 'count'),
    pct_prioridad_alta=('prioridad', lambda s: (s == 'alta').mean() * 100),
    edad_media_paciente=('paciente_edad', 'mean'),
    tiempo_respuesta_medio_min=('tiempo_de_respuesta', 'mean'),
).round(2).reset_index()

print(f"Distritos con activaciones: {len(demanda)}")
demanda.head()

Activaciones con distrito asignado: 145966 de 146567 (quedan fuera 601, el 0.41%)
Distritos con activaciones: 21


,distrito,n_activaciones,pct_prioridad_alta,edad_media_paciente,tiempo_respuesta_medio_min
0,ARGANZUELA,7849,39.18,49.27,6.81
1,BARAJAS,2210,48.05,50.70,10.72
2,CARABANCHEL,10421,34.95,49.38,8.06
3,CENTRO,21182,35.70,50.07,7.20
4,CHAMARTIN,6785,43.85,49.02,7.17


## 3. Qué aportan los demás datasets

Bluetab nos dijo que mirásemos otros datasets para cruzar información. Esto es lo que hemos encontrado, dataset a dataset.

In [4]:
uni = cargar('unidades_samur_limpio.csv')
bases = cargar('bases_samur_limpio.csv')
hosp = cargar('hospitales_limpio.csv')
acc = cargar('accidentes.csv')

# UNIDADES SAMUR (catalogo limpio): encaja por unidad_id -> carga de trabajo por unidad y por base.
match_uni = act['unidad_id'].isin(set(uni['unidad_id'])).mean() * 100
print(f"1. unidades_samur: encaja el {match_uni:.0f}% por unidad_id (catalogo limpio, {len(uni)} unidades).")

# TIPOS DE EMERGENCIA: resuelto en la limpieza de activaciones, no hace falta cruzar aqui.
tipos_clean = cargar('tipos_emergencia_clean.csv')
match_clean = act['tipo_emergencia'].isin(set(tipos_clean['codigo']))
print(f"2. tipos_emergencia: activaciones_limpio.csv ya trae tipo_emergencia = codigo del catalogo "
      f"({len(tipos_clean)} entradas) y prioridad = la del catalogo. Coincidencia directa: "
      f"{match_clean.mean()*100:.1f}% ({int(match_clean.sum())} de {len(act)}). "
      f"Categoria, tiempo objetivo y si requiere medico se unen por esa misma columna cuando hagan falta.")

# BASES Y HOSPITALES (catalogos limpios): catalogos por distrito -> oferta sanitaria.
print(f"3. bases_samur: {len(bases)} bases; hospitales: {len(hosp)} hospitales (catalogos limpios). "
      f"Se cruzan por distrito (normalizando el nombre).")

# ACCIDENTES: trafico, con municipios de fuera de Madrid y distritos que faltan.
en_catalogo = acc['distrito'].map(normalizar_texto).isin(set(dis['nombre'])).sum()
print(f"4. accidentes: solo {en_catalogo} de {len(acc)} filas tienen un distrito de Madrid ciudad; "
      f"el resto son municipios de fuera o nombres ambiguos.")

# METEOROLOGIA Y CALENDARIO: van por dia y activaciones no tiene dia.
print("5. meteorologia y calendario: van por dia. Activaciones solo tiene año, mes y hora, "
      "asi que no se pueden cruzar por fecha.")

1. unidades_samur: encaja el 100% por unidad_id (catalogo limpio, 60 unidades).
2. tipos_emergencia: activaciones_limpio.csv ya trae tipo_emergencia = codigo del catalogo (39 entradas) y prioridad = la del catalogo. Coincidencia directa: 100.0% (146567 de 146567). Categoria, tiempo objetivo y si requiere medico se unen por esa misma columna cuando hagan falta.
3. bases_samur: 15 bases; hospitales: 20 hospitales (catalogos limpios). Se cruzan por distrito (normalizando el nombre).
4. accidentes: solo 1596 de 2062 filas tienen un distrito de Madrid ciudad; el resto son municipios de fuera o nombres ambiguos.
5. meteorologia y calendario: van por dia. Activaciones solo tiene año, mes y hora, asi que no se pueden cruzar por fecha.


### Lo que decidimos con cada uno

- **unidades_samur: sí.** Encaja el 100% por `unidad_id` (unir por id, no por el código, que se repite). Lo usamos para la oferta: cuántas unidades tiene cada distrito en su base. Ojo: con el catálogo limpio hay 10 unidades cuya base pone `VALLECAS` a secas y 5 con base `NO CONSTA` o vacía. Esas 15 no se reparten entre distritos, porque no hay forma de saber a cuál pertenecen (ver sección 4).
- **bases_samur y hospitales: sí.** Se cruzan por distrito y dan la oferta sanitaria: bases, unidades, hospitales y camas. El caso pregunta justo por la relación entre densidad y recursos del SAMUR. Los catálogos limpios ya traen los nombres de distrito casi listos, y aplicamos la misma regla de normalización de siempre (ver más abajo). Además comparamos sus conteos con las columnas `num_hospitales` y `base_samur` de `distritos.csv`, y no coinciden en varios distritos (ver la comprobación de abajo). Y un cambio importante respecto a la primera versión: las camas que no constan en el catálogo se quedan **vacías, sin imputar** (antes se les ponía la media, y eso era inventar un dato).
- **tipos_emergencia: sí, ya resuelto en la limpieza de activaciones.** Antes no se unía el catálogo porque `INTOXICACION ETILICA` y `CAIDA` salían dos veces con categoría y prioridad distintas, sin forma de saber cuál valía. Eso se corrigió en el catálogo limpio (39 entradas, una por concepto) y `limpieza_activaciones.ipynb` ya deja `tipo_emergencia` con el código del catálogo y `prioridad` con la del catálogo, así que aquí no hay que cruzar nada: la categoría, el tiempo objetivo y si requiere médico se traen con un `merge` directo por esa columna cuando haga falta. El mix de tipos y su asociación con renta y vulnerabilidad se calcula en el notebook socioeconómico definitivo.
- **accidentes: no para el clustering.** Es de tráfico, incluye municipios de fuera de Madrid (Alcorcón, Getafe, Móstoles...) que no están en el socioeconómico, y le faltan distritos. Se queda como posible ampliación (riesgo vial por distrito).
- **meteorología y calendario: no se puede.** Van por día y las activaciones no tienen día: el dato abierto del SAMUR solo publica año, mes y horas (lo confirmó Bluetab por correo el 14/10), así que es una limitación del dato. Cruzar solo por año-mes sería una asociación demasiado gruesa para el clustering.
- **Vallecas a secas (39 registros de origen, antes de filtros):** Bluetab sugirió mirar si otros datasets lo resolvían. Probamos unidades_samur y hospitales y no distinguen entre Puente y Villa de Vallecas; accidentes, meteorología y calendario no comparten clave con esas filas. El detalle está en `limpieza_activaciones.ipynb` (sección "Vallecas a secas") y la decisión es mantenerlas como NO CONSTA.

El alcance exacto de las fuentes primarias está identificado al inicio. Las ventanas concretas de medianoche y la decisión de no imputar son propias; no se atribuye al correo la aprobación de cada valor del catálogo.

## 4. Tabla final y límites de la oferta

La tabla tiene 21 filas y22 columnas. Los dos distritos hospitalarios se corrigen con evidencia en `limpieza_hospitales.ipynb`: Infanta Leonor pasa a Villa de Vallecas y Jiménez Díaz a Moncloa-Aravaca. Son correcciones de ubicación del recurso, nunca de residencia o lugar de activación.

Decisiones definitivas:
- Las 10 unidades con base VALLECAS no se reparten; las 5 NO CONSTA tampoco. No existe correspondencia demostrable entre sus IDs locales y una base oficial. Puente y Villa quedan desconocidos en unidades; no se presupone que sean0.
- Los 9 hospitales de otros municipios quedan fuera del agregado por distrito de Madrid.
- Se cuentan los hospitales del catálogo con ubicación asignada, sin llamarlos inventario completo de la ciudad.
- No se mezclan cifras de camas de otros años/definiciones. Si falta el dato de un hospital del distrito, el agregado de camas queda vacío: suma parcial no es capacidad total.
- Un0 indica ausencia de registros en este catálogo, **no ausencia real de recurso**. Las 3 columnas de oferta `unidades_samur`, `num_hospitales_cat` y `camas_hospital` quedan excluidas del clustering por alcance: el modelo describe perfiles y servicio, no capacidad operativa; además unidades y camas tienen las limitaciones anteriores. Su exclusión no depende de una confirmación futura.

La tabla conserva todas las columnas para trazabilidad; las matrices de 9 variables principales y 10 de comparación se comprueban en el notebook socioeconómico.


In [5]:
# Oferta por distrito: bases, unidades y hospitales (catalogos limpios)
oficiales = set(dis['nombre'])

def completar_distrito(texto, candidatos, contiene=False):
    t = normalizar_texto(texto)
    if t is None:
        return None
    for articulo in ('LA ', 'EL ', 'LOS ', 'LAS '):
        if t.startswith(articulo):
            t = t[len(articulo):]
    if t in candidatos:
        return t
    # principio de un nombre (o texto contenido) y solo si es uno
    posibles = [c for c in candidatos if (t in c if contiene else c.startswith(t))]
    return posibles[0] if len(posibles) == 1 else None

bases['distrito_norm'] = bases['distrito'].apply(lambda x: completar_distrito(x, oficiales))
hosp['distrito_norm'] = hosp['distrito'].apply(lambda x: completar_distrito(x, oficiales))
uni['distrito_norm'] = uni['base_distrito'].apply(lambda x: completar_distrito(x, oficiales))

print("Bases sin distrito oficial:", bases.loc[bases['distrito_norm'].isna(), 'distrito'].tolist())
print("Hospitales sin distrito de Madrid:", hosp.loc[hosp['distrito_norm'].isna(), 'distrito'].tolist())
print("Unidades sin distrito asignable:", int(uni['distrito_norm'].isna().sum()), "de", len(uni),
      "(10 con base VALLECAS, que no distingue Puente/Villa, y 5 con base NO CONSTA o vacia)")
print()

oferta_bases = bases.groupby('distrito_norm').agg(num_bases_samur=('base_id', 'count')).reset_index()
oferta_unidades = uni.groupby('distrito_norm').agg(unidades_samur=('unidad_id', 'count')).reset_index()
oferta_hosp = hosp.groupby('distrito_norm').agg(
    num_hospitales_cat=('hospital_id', 'count'), camas_hospital=('camas', lambda s: s.sum() if s.notna().all() else np.nan)).reset_index()

tabla = (base
         .merge(demanda, left_on='nombre', right_on='distrito', how='left')
         .merge(oferta_bases, left_on='nombre', right_on='distrito_norm', how='left')
         .merge(oferta_unidades, left_on='nombre', right_on='distrito_norm', how='left')
         .merge(oferta_hosp, left_on='nombre', right_on='distrito_norm', how='left'))

tabla['activaciones_por_1000hab'] = (tabla['n_activaciones'] / tabla['poblacion'] * 1000).round(2)

# Ceros reales: un distrito que no aparece en un catalogo no tiene ese recurso segun el catalogo.
# Excepcion: un hospital que si sale pero sin dato de camas no es 0 camas, es desconocido (vacio).
tabla['num_bases_samur'] = tabla['num_bases_samur'].fillna(0).astype(int)
tabla['unidades_samur'] = tabla['unidades_samur'].fillna(0)
tabla['num_hospitales_cat'] = tabla['num_hospitales_cat'].fillna(0)
tabla.loc[tabla['num_hospitales_cat'] == 0, 'camas_hospital'] = tabla.loc[tabla['num_hospitales_cat'] == 0, 'camas_hospital'].fillna(0)

# Solo las unidades siguen siendo ambiguas entre Puente y Villa.
# El hospital Infanta Leonor tiene ubicación oficial en Villa; no se anulan sus agregados.
VALLECAS_DUDOSOS = ['PUENTE DE VALLECAS', 'VILLA DE VALLECAS']
tabla.loc[tabla['nombre'].isin(VALLECAS_DUDOSOS), 'unidades_samur'] = np.nan

print("Celdas vacias en la oferta (desconocido, no cero):")
for c in ['unidades_samur', 'num_hospitales_cat', 'camas_hospital']:
    falta = tabla.loc[tabla[c].isna(), 'nombre'].tolist()
    print(f"  {c}: {falta}")

cols_tabla = ['distrito_id', 'nombre', 'poblacion', 'superficie_km2', 'densidad_hab_km2',
              'renta_media_hogar', 'edad_media', 'pct_mayores_65', 'pct_extranjeros',
              'tasa_paro', 'indice_vulnerabilidad', 'num_centros_salud', 'parques_m2_habitante',
              'n_activaciones', 'activaciones_por_1000hab', 'pct_prioridad_alta',
              'edad_media_paciente', 'tiempo_respuesta_medio_min', 'num_bases_samur', 'unidades_samur',
              'num_hospitales_cat', 'camas_hospital']
tabla = tabla[cols_tabla].rename(columns={'nombre': 'distrito'})
for c in ['unidades_samur', 'num_hospitales_cat', 'camas_hospital']:
    tabla[c] = tabla[c].astype('Int64')   # enteros con vacio permitido
tabla.to_csv(carpeta / 'tabla_distritos.csv', index=False)
print(tabla.shape)   # (21, 22)
tabla.head(10)

Bases sin distrito oficial: []
Hospitales sin distrito de Madrid: ['GETAFE', 'LEGANES', 'ALCALA DE HENARES', 'MOSTOLES', 'COSLADA', 'SAN SEBASTIAN DE LOS REYES', 'MOSTOLES', 'PARLA', 'ALCORCON']
Unidades sin distrito asignable: 15 de 60 (10 con base VALLECAS, que no distingue Puente/Villa, y 5 con base NO CONSTA o vacia)

Celdas vacias en la oferta (desconocido, no cero):
  unidades_samur: ['PUENTE DE VALLECAS', 'VILLA DE VALLECAS']
  num_hospitales_cat: []
  camas_hospital: ['MONCLOA-ARAVACA']
(21, 22)


,distrito_id,distrito,poblacion,superficie_km2,densidad_hab_km2,renta_media_hogar,edad_media,pct_mayores_65,pct_extranjeros,tasa_paro,...,parques_m2_habitante,n_activaciones,activaciones_por_1000hab,pct_prioridad_alta,edad_media_paciente,tiempo_respuesta_medio_min,num_bases_samur,unidades_samur,num_hospitales_cat,camas_hospital
0,1,CENTRO,149718,5.23,28626.8,42831.0,44.2,22.1,21.5,8.2,...,3.2,21182,141.48,35.70,50.07,7.20,1,3,0,0
1,2,ARGANZUELA,155115,6.47,23974.5,39876.0,42.8,18.5,18.2,9.5,...,8.5,7849,50.60,39.18,49.27,6.81,1,0,0,0
2,3,RETIRO,119915,5.46,21962.5,61604.0,47.3,25.3,12.3,6.8,...,25.6,5184,43.23,45.58,49.14,6.54,0,0,1,1500
3,4,SALAMANCA,147707,5.41,27302.6,71432.0,48.1,26.8,11.5,5.2,...,4.8,7523,50.93,45.21,49.33,6.51,0,0,1,480
4,5,CHAMARTIN,146512,9.19,15942.5,62145.0,46.5,25.2,10.8,5.8,...,6.2,6785,46.31,43.85,49.02,7.17,1,0,1,150
5,6,TETUAN,155159,5.37,28893.7,35678.0,41.2,17.8,22.4,11.2,...,2.8,7957,51.28,36.24,49.94,6.88,1,11,0,0
6,7,CHAMBERI,137532,4.68,29387.2,51234.0,45.8,23.5,13.5,7.5,...,3.5,7001,50.90,38.09,49.49,6.48,0,0,0,0
7,8,FUENCARRAL-EL PARDO,245974,237.84,1034.2,48765.0,42.1,16.2,12.1,8.1,...,45.2,6099,24.80,46.78,49.52,9.49,1,6,1,1328
8,9,MONCLOA-ARAVACA,123606,46.29,2670.3,55432.0,43.5,19.8,11.2,7.6,...,18.5,7995,64.68,41.71,49.07,7.68,1,0,3,<NA>
9,10,LATINA,233765,25.37,9214.2,31245.0,43.8,20.5,18.5,12.5,...,5.8,8314,35.57,36.73,49.55,8.11,1,11,0,0


### Comprobación: `distritos.csv` frente a los catálogos

En `distritos.csv` vienen `num_hospitales` y `base_samur` por distrito. Los comparamos con lo que sale de contar los catálogos limpios de hospitales y bases. No corregimos ninguna de las dos fuentes: solo vemos dónde no coinciden.

In [6]:
comparacion = dis[['nombre', 'num_hospitales', 'base_samur']].merge(
    oferta_hosp.rename(columns={'distrito_norm': 'nombre'})[['nombre', 'num_hospitales_cat']], on='nombre', how='left').merge(
    oferta_bases.rename(columns={'distrito_norm': 'nombre'}), on='nombre', how='left')
comparacion[['num_hospitales_cat', 'num_bases_samur']] = comparacion[['num_hospitales_cat', 'num_bases_samur']].fillna(0)
comparacion['base_catalogo'] = (comparacion['num_bases_samur'] > 0).astype(int)

dif_hosp = comparacion[comparacion['num_hospitales'] != comparacion['num_hospitales_cat']]
dif_base = comparacion[comparacion['base_samur'] != comparacion['base_catalogo']]
print(f"Hospitales: no coinciden en {len(dif_hosp)} de 21 distritos")
print(dif_hosp[['nombre', 'num_hospitales', 'num_hospitales_cat']].to_string(index=False))
print()
print(f"Base SAMUR: no coincide en {len(dif_base)} de 21 distritos")
print(dif_base[['nombre', 'base_samur', 'base_catalogo']].to_string(index=False))

Hospitales: no coinciden en 14 de 21 distritos
             nombre  num_hospitales  num_hospitales_cat
             CENTRO               3                 0.0
         ARGANZUELA               2                 0.0
          SALAMANCA               2                 1.0
          CHAMARTIN               3                 1.0
             TETUAN               1                 0.0
           CHAMBERI               2                 0.0
FUENCARRAL-EL PARDO               2                 1.0
    MONCLOA-ARAVACA               2                 3.0
             LATINA               1                 0.0
 PUENTE DE VALLECAS               1                 0.0
      CIUDAD LINEAL               2                 0.0
         VILLAVERDE               1                 0.0
  VILLA DE VALLECAS               0                 1.0
SAN BLAS-CANILLEJAS               1                 0.0

Base SAMUR: no coincide en 6 de 21 distritos
         nombre  base_samur  base_catalogo
         TETUAN         

## 5. Primer vistazo: renta, vulnerabilidad y demanda

El caso pregunta si los barrios más vulnerables tienen una demanda distinta. Como primer vistazo (sin sacar conclusiones todavía, que eso es el clustering), miramos las correlaciones entre la demanda por 1.000 habitantes y los indicadores sociales, y los tipos de emergencia más frecuentes.

In [7]:
cols_corr = ['activaciones_por_1000hab', 'renta_media_hogar', 'indice_vulnerabilidad',
             'pct_mayores_65', 'tasa_paro', 'densidad_hab_km2', 'edad_media_paciente']
print("Correlaciones con la demanda por 1.000 habitantes:")
print(tabla[cols_corr].corr()['activaciones_por_1000hab'].round(2).sort_values())
print()
print("Tipos de emergencia mas frecuentes (top 10):")
print(act['tipo_emergencia'].value_counts().head(10).to_string())
print()
tabla.describe().round(2)

Correlaciones con la demanda por 1.000 habitantes:
indice_vulnerabilidad      -0.27
tasa_paro                  -0.27
renta_media_hogar           0.19
edad_media_paciente         0.24
pct_mayores_65              0.31
densidad_hab_km2            0.46
activaciones_por_1000hab    1.00
Name: activaciones_por_1000hab, dtype: float64

Tipos de emergencia mas frecuentes (top 10):
tipo_emergencia
Patología cardiovascular         28715
Casual: caída, etc               26265
Intoxicación etílica             14098
Agresión sin especificar          8608
Accidente menos de 3 victimas     7990
Heridas                           6756
Patología psicosomática           5807
Certificación psiquiátrico        5359
Otros                             5236
Accidente de motocicleta          4674



,distrito_id,poblacion,superficie_km2,densidad_hab_km2,renta_media_hogar,edad_media,pct_mayores_65,pct_extranjeros,tasa_paro,indice_vulnerabilidad,...,parques_m2_habitante,n_activaciones,activaciones_por_1000hab,pct_prioridad_alta,edad_media_paciente,tiempo_respuesta_medio_min,num_bases_samur,unidades_samur,num_hospitales_cat,camas_hospital
count,21.0,21.00,21.00,21.00,21.00,21.00,21.00,21.00,21.00,21.00,...,21.00,21.00,21.00,21.00,21.00,21.00,21.00,19.0,21.0,20.0
mean,11.0,157604.19,28.80,14410.11,41709.81,42.95,18.95,16.55,10.54,0.46,...,11.97,6950.76,45.37,40.87,49.49,8.02,0.71,2.37,0.52,328.3
std,6.2,56809.83,50.08,9945.07,13538.19,2.74,4.19,4.58,3.66,0.20,...,11.51,3945.80,24.21,4.83,0.45,1.22,0.46,3.7,0.75,523.07
min,1.0,47703.00,4.68,1034.20,23456.00,37.20,11.20,10.50,5.20,0.15,...,2.80,2210.00,24.80,32.44,48.67,6.48,0.00,0.0,0.0,0.0
25%,6.0,123606.00,6.31,6838.00,31245.00,41.50,16.20,12.30,7.60,0.32,...,4.50,4943.00,33.94,36.73,49.24,7.17,0.00,0.0,0.0,0.0
50%,11.0,149718.00,14.05,15942.50,39876.00,42.80,18.50,16.80,9.80,0.45,...,6.50,6748.00,40.50,39.89,49.49,7.76,1.00,0.0,0.0,0.0
75%,16.0,193104.00,28.24,21962.50,51234.00,44.20,22.10,20.80,12.80,0.58,...,15.80,7957.00,50.60,45.21,49.68,8.58,1.00,4.5,1.0,485.0
max,21.0,257329.00,237.84,29387.20,71432.00,48.10,26.80,25.60,17.20,0.82,...,45.20,21182.00,141.48,49.17,50.70,10.72,1.00,11.0,3.0,1500.0


## 6. Normalización0-1

Cada columna numérica se escala con `(valor-min)/(max-min)` y se redondea a 4 decimales; una columna constante se deja en 0 porque no distingue distritos. Los originales quedan aparte. Los huecos de unidades y camas se conservan, sin imputación. Las columnas de oferta siguen fuera del clustering por la decisión justificada en sección4, aunque alguna esté completa. El notebook final comprueba que las 9 entradas principales y las 10 de comparación son finitas y que su min-max coincide con el CSV normalizado.


In [8]:
tabla_guardada = pd.read_csv(carpeta / 'tabla_distritos.csv')
cols_norm_tabla = [c for c in tabla_guardada.columns if c not in ('distrito_id', 'distrito')]
tabla_norm = tabla_guardada[['distrito_id', 'distrito']].copy()
for c in cols_norm_tabla:
    minimo, maximo = tabla_guardada[c].min(), tabla_guardada[c].max()
    if maximo > minimo:
        tabla_norm[c] = ((tabla_guardada[c] - minimo) / (maximo - minimo)).round(4)
    else:
        tabla_norm[c] = 0.0   # columna constante: no distingue distritos
tabla_norm.to_csv(carpeta / 'tabla_distritos_norm.csv', index=False)
print(tabla_norm.shape)   # (21, 22): mismas columnas, valores entre 0 y 1
vacios = tabla_norm.isna().sum()
print("Vacios por columna (desconocido conservado, no imputado):")
print(vacios[vacios > 0].to_string())
print("La oferta queda fuera del clustering: decision definitiva del analisis, no imputacion.")
tabla_norm.describe().round(2)

(21, 22)
Vacios por columna (desconocido conservado, no imputado):
unidades_samur    2
camas_hospital    1
La oferta queda fuera del clustering: decision definitiva del analisis, no imputacion.


,distrito_id,poblacion,superficie_km2,densidad_hab_km2,renta_media_hogar,edad_media,pct_mayores_65,pct_extranjeros,tasa_paro,indice_vulnerabilidad,...,parques_m2_habitante,n_activaciones,activaciones_por_1000hab,pct_prioridad_alta,edad_media_paciente,tiempo_respuesta_medio_min,num_bases_samur,unidades_samur,num_hospitales_cat,camas_hospital
count,21.0,21.00,21.00,21.00,21.00,21.00,21.00,21.00,21.00,21.00,...,21.00,21.00,21.00,21.00,21.00,21.00,21.00,19.00,21.00,20.00
mean,11.0,0.52,0.10,0.47,0.38,0.53,0.50,0.40,0.45,0.46,...,0.22,0.25,0.18,0.50,0.40,0.36,0.71,0.22,0.17,0.22
std,6.2,0.27,0.21,0.35,0.28,0.25,0.27,0.30,0.31,0.29,...,0.27,0.21,0.21,0.29,0.22,0.29,0.46,0.34,0.25,0.35
min,1.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,6.0,0.36,0.01,0.20,0.16,0.39,0.32,0.12,0.20,0.25,...,0.04,0.14,0.08,0.26,0.28,0.16,0.00,0.00,0.00,0.00
50%,11.0,0.49,0.04,0.53,0.34,0.51,0.47,0.42,0.38,0.45,...,0.09,0.24,0.13,0.45,0.40,0.30,1.00,0.00,0.00,0.00
75%,16.0,0.69,0.10,0.74,0.58,0.64,0.70,0.68,0.63,0.64,...,0.31,0.30,0.22,0.76,0.50,0.50,1.00,0.41,0.33,0.32
max,21.0,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,...,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
